---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: APIs

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.



---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 🚪 1. Introduction

## What's an API?

An API, or Application Programming Interface, is a set of protocols and tools that allow different software applications to communicate with each other. 
- **It can be thought of as a contract**, where one software promises to provide certain functionalities (through endpoints) in response to a specific set of input parameters.
- Think of an API as a menu in a restaurant. The menu provides a list of dishes you can order, along with a description of each dish. When you specify what menu items you want, the restaurant's kitchen does the work and provides you with some finished dishes. You don't know exactly how the restaurant prepares that food, and you don't really need to. 
- Similarly, an API lists a bunch of operations that developers can use, along with a description of what they do. The developer doesn't necessarily need to know how, for example, an operating system builds and manages a file system; they just need to know that there's an API call that can be used to list files in a directory.

Pretty much every single command you use can be thought of as an API.

## Why are APIs important?
APIs play a pivotal role in today's digital world. 
- They allow for the integration of systems, making it possible for applications to share data and functionalities seamlessly. 
- They allow for scalability by decoupling server-side functionality from the client side
- They allow easy access to third-party tools and functionalities


## Accessing APIs
APIs can be:

1. **Open/Public**: Available to outside developers. Public does not mean unlimited or free: an API can still require a key, payment, and compliance with rate limits and terms. 
2. **Partner**: These APIs are typically offered to a select number of partners, usually through an agreement. They may require some form of authentication or access key.
3. **Internal**: These APIs are usually not exposed to the public. They are used to build internal systems and functionalities.

## API Architecture

There are different ways to design an API. Three names you'll run into:

1. **REST**: An architectural style organized around resources, such as `/breeds/123`. Web APIs following this style commonly use HTTP methods like GET to read data and POST to create it. Stateless requests are one of REST's constraints.
2. **GraphQL**: A query language and runtime for APIs, with a published [specification](https://spec.graphql.org/). The client can ask for particular fields instead of accepting a fixed response shape.
3. **gRPC**: A [remote procedure call framework](https://grpc.io/docs/what-is-grpc/introduction/). You call a method on another computer rather than a local Python function. It commonly uses Protocol Buffers for messages and HTTP/2 for transport.

These names don't promise that every request is cacheable or that the same call always returns the same answer. The data can change! Check the API's documentation for how it behaves. We'll focus on HTTP APIs returning JSON.

---
# 2. Making API Calls

We'll start with your old favorite, `requests`. It handles the details of sending an HTTP request; we still need to check whether that request worked and understand the response.

## Example 1 - dog facts
Here's a public REST API that doesn't require a key. `params` adds query parameters to the URL; here, we're asking for two facts. `timeout` keeps us from waiting indefinitely if the site doesn't respond.

In [ ]:
import requests

response = requests.get(
    'https://dogapi.dog/api/v2/facts', params={'limit': 2}, timeout=30
)

In [ ]:
# Print the status code of the response.
response.status_code

In [ ]:
# Ensure the request was successful
response.raise_for_status()

In [ ]:
response.json()

In [ ]:
# The response contains a list under 'data'; each fact has nested attributes
facts = response.json()['data']
for fact in facts:
    print(fact['attributes']['body'])
    print()

## Example 2 - Star Wars

In [ ]:
response = requests.get("https://swapi.dev/api/people/1", timeout=30)
response.raise_for_status()
response.json()

In [ ]:
response = requests.get("https://swapi.dev/api/people/2", timeout=30)
response.raise_for_status()
response.json()

In [ ]:
response = requests.get("https://swapi.dev/api/planets/1", timeout=30)
response.raise_for_status()
response.json()

In [ ]:
type(response.json())

In [ ]:
response = requests.get("https://swapi.dev/api/starships/2", timeout=30)
response.raise_for_status()
response.json()

---
---

## Exercise # 1

Take a look at the [Dog API documentation](https://dogapi.dog/docs/api-v2).

1. What is the name of the breed with id `a6ea38ed-f692-478e-af29-378d0e2cc270`?
2. For the dog in step 1, what is the name of the group that it belongs to?
3. What are the names of all breeds that belong to the group in step 2?

Hint: `attributes` contains details about a resource, while `relationships` links it to other resources by id. An id isn't the name: you may need another request to look it up. If an endpoint returns pages, check `links['next']` rather than assuming the first page contains everything. Keep `timeout=30` and `raise_for_status()` in your requests, just as above.

---
---

# 3. The OpenAI Chat Completions Endpoint

## 3.1 Overhead

Use `09.1 openai-api.ipynb` before class to set up your key. The required packages were installed when you ran `uv sync`.

Keep your key in the `.env` file in your `wa` folder, never in a notebook. This is paid API access, separate from a ChatGPT subscription. Run each request once to start; rerunning a cell makes another paid request. If something fails, read the error before repeatedly trying again.

In [ ]:
# OpenAI and tiktoken were installed when you ran `uv sync`.


Set up imports, API key, and client for this section.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Same course-folder setup as in the other notebooks
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')
data_folder = course_folder / 'data'

In [ ]:
# Load our course .env file; an existing environment variable can also supply the key
load_dotenv(course_folder / '.env')
my_api_key = os.getenv('OPENAI_API_KEY')

if not my_api_key or not my_api_key.strip():
    raise RuntimeError(
        'No OpenAI API key found. Add OPENAI_API_KEY to the .env file in your wa folder, '
        'save the file, then rerun this cell. Do not paste your key into this notebook.'
    )

# Check that we found a key WITHOUT printing it
print('Successfully retrieved OpenAI API key!')

In [ ]:
# The SDK handles the HTTP requests for us
# Stop waiting after 30 seconds; don't automatically retry failed paid requests
client = OpenAI(api_key=my_api_key, timeout=30, max_retries=0)

# One small, inexpensive model throughout, just as in the setup notebook
model_name = 'gpt-5.4-nano'

---
## 3.2 Making a request

In the before-class notebook, we made our first request to the OpenAI API. Let's break that process down.

- We're making a request to the chat completions endpoint, as indicated by `chat.completions`. 
- We send a conversation to a model, somewhat like the conversations we have in ChatGPT.
- It takes a message (or list of messages) provided by the user and it returns a message generated by the model as output.

In [ ]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{'role': 'user', 'content': 'Tell me one short fun fact about cats.'}],
    reasoning_effort='none',
    max_completion_tokens=128,
)

print(response.choices[0].message.content)

In [ ]:
# Ask for JSON, then actually turn it into a Python dictionary
review = 'The story dragged, and the characters never felt believable. I would not recommend it.'
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {'role': 'system', 'content': 'Classify the book review. Return a JSON object with '
         'classification (positive or negative) and reason (a brief explanation).'},
        {'role': 'user', 'content': review},
    ],
    response_format={'type': 'json_object'},
    reasoning_effort='none',
    max_completion_tokens=200,
)

if response.choices[0].finish_reason != 'stop':
    raise ValueError('The model did not finish its answer. Inspect the response before using it.')

message = response.choices[0].message
if message.refusal or not message.content:
    raise ValueError('No classification returned. Inspect the response before using it.')
result = json.loads(message.content)
if not isinstance(result, dict):
    raise ValueError('Expected a JSON object.')
if not isinstance(result.get('classification'), str) or result['classification'] not in {'positive', 'negative'}:
    raise ValueError('Expected positive or negative for the classification.')
if not isinstance(result.get('reason'), str) or not result['reason'].strip():
    raise ValueError('Expected a short explanation in reason.')

print(result['classification'])
print(result['reason'])

In [ ]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{'role': 'user', 'content': 'Write a Python function that takes a list of numbers '
               'and returns their mean. Include one example. Keep it short.'}],
    reasoning_effort='none',
    max_completion_tokens=250,
)

# Read the code before deciding whether to run it; we aren't executing it here
print(response.choices[0].message.content)

Two important parameters in these requests are `model` and `messages`.

- `model`: the model we want to use. We've put its name in `model_name`, so we can change it in one place. See the [model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-nano).
- `messages`: a list of dictionaries describing the conversation. In our examples we use three roles:
  - `system`: instructions about how the model should behave.
  - `user`: the message we send it.
  - `assistant`: an answer from the model, or an example answer we include in the conversation.

In the classification example, JSON mode asks for valid JSON. It doesn't guarantee the fields or labels we wanted, so we still check them. A valid JSON answer can also be wrong about the review! If you're curious, [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs) can enforce a schema; we don't need that extra machinery for this example.

Here's an example of how we can use the `messages` parameter to simulate a conversation between the user and the model, giving the model an example of the type of output we want:

In [ ]:
response = client.chat.completions.create(
  model=model_name,
  messages=[
    { 
      "role": "system", 
      "content": "You are a fantastic film critic. You rate movies using simple terms, only outputting the word 'negative' or 'positive'."
    },
    {
      "role": "user", 
      "content": "Wonderful spectacle, terrific acting and toweringly great film-making."
    },
    {
      "role": "assistant", 
      "content": "positive"
    },
    {
      "role": "user", 
      "content": "A disappointing entry in the Scorsese canon."
    },
    {
      "role": "assistant", 
      "content": "negative"
    },
    {
      "role": "user", 
      "content": "Gangs Of New York is a magnificent film, and is challengingly about more than the sum of its parts."
    }
  ],
  reasoning_effort='none',
  max_completion_tokens=32,
)

print(response.choices[0].message.content)

### 3.3 Optional parameters

We can also control parts of the request:

- `max_completion_tokens`: caps the generated tokens, including reasoning tokens if the model uses them. A low cap can cut an answer short.
- `reasoning_effort`: supported by our model; we use `'none'` for these simple examples.
- `n`: the number of alternative completions. We stick to 1 to keep costs down.

You may see `temperature` and `top_p` in other examples. They control sampling, but support depends on the model and settings. Lower temperature doesn't guarantee a correct or perfectly repeatable answer. Don't copy parameters into a different model without checking its documentation.

Details: [Chat Completions reference](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create).

In [ ]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{'role': 'user', 'content': 'Tell me one short fun fact about cats.'}],
    reasoning_effort='none',
    max_completion_tokens=128,
    n=1,
)

print(response.choices[0].message.content)

---
## 3.4 The response object

So far, we've used this to print the model's answer:

```python
print(response.choices[0].message.content)
```

But what is everything around `content`? Let's inspect the response we already have. No need to pay for another request just to look at it!

In [ ]:
# Reuse the cat-fact response from the previous cell
# Inspecting an existing Python object does not make another API request

The API returns a `ChatCompletion` object:

In [ ]:
print(type(response))

In [ ]:
print(response)

For our purposes, the `ChatCompletion` object is somewhat like a nested dictionary. 

Instead of using bracket notation to navigate (e.g., `response["choices"]`), we use dot notation, like in the code below:

In [ ]:
print(response.choices)

> **Tip:** `response` and each choice are SDK objects, so we use dots for their attributes. `response.choices` is a Python list, so we use `[0]` to select its first item. These are the same objects-versus-lists ideas we've used elsewhere in Python.

`response.choices` is a list of outputs. We asked for one cat fact (`n=1`), so it contains one choice. We can loop through the choices without hard-coding the list length:

In [ ]:
for choice in response.choices:
    print('Message content:', choice.message.content)
    print('Role:', choice.message.role)
    print('Finish reason:', choice.finish_reason)
    print('-----------------------------------------------------')

We are generally interested in the message content. However, as we see above, the API also gives us some metadata. For instance, `finish_reason` tells us why the model stopped generating output (e.g. it hit a natural stopping point, it ran out of tokens, etc.).

In the next section, we'll look at one aspect of the metadata that is of practical interest: tokenization.

---
## 3.5 Tokenization

### 3.5.1 What is a token?

When you make a request to the OpenAI API, your natural language inputs are converted into tokens. Similarly, the model also returns tokens in its output. 

A token is a common sequence of characters. Usually, a token is either a word or a word fragment, but it can also contain digits and symbols.

We can use the [online OpenAI tokenizer](https://platform.openai.com/tokenizer) to see how text maps to tokens. Here's an example:

<div style="text-align:center;">
  <img src="https://github.com/apostolosfilippas/wa/blob/50885e30251cc4175d2d895a5bb895bb525107b7/assets/api-1.png?raw=true" 
    width="700">
</div>

In the example above, we observe that:

- Some words map directly to tokens, like "how" and "use"
- Other words, like "we're" and "OpenAI," are broken up into multiple tokens
- The period at the end of the sentence is also a token by itself

### 3.5.2 Tokenizing in Python

It's useful to compute the number of tokens directly in Python. We can do this using the `tiktoken` package created by OpenAI.

Calling `encoding.encode` will give us a list of ids. Each id corresponds to a token. We'll select `o200k_base`, the [encoding used by the GPT-5 family](https://github.com/openai/tiktoken/blob/main/tiktoken/model.py), explicitly because older versions of tiktoken might not recognize the newer model name. The first run may download the encoding file; after that, tiktoken reuses its cache:

In [ ]:
import tiktoken

sentence = "We're learning how to use the OpenAI API in class today."

# GPT-5.4 nano uses o200k_base; select it explicitly for older tiktoken versions
encoding = tiktoken.get_encoding('o200k_base')

sentence_encoding = encoding.encode(sentence)

print(sentence_encoding)

We can count the number of tokens by getting the length of the list:

In [ ]:
len(sentence_encoding)

We can also convert our list of ids back into natural language by calling `encoding.decode`:

In [ ]:
encoding.decode(sentence_encoding)

### 3.5.3 Why do we care?

- **Billing:** Input and output tokens have different prices. More requests and longer prompts can add up.
- **Model constraints:** Models have limits on input, output, and total context. These depend on the model; don't assume that any size of book or document will fit.
- **Time:** Longer answers generally take longer to generate. `max_completion_tokens` puts a ceiling on generated tokens.

The token count we computed for one sentence isn't the full bill for a chat request: the messages and other request formatting also take tokens. Use the API's returned `usage` for that request.

Let's look at the token usage from our last request:

In [ ]:
print(response.choices[0].message.content)

# Tokens from our input messages (the prompt)
print("Prompt tokens:", response.usage.prompt_tokens)

# Tokens from the model's response
print("Completion tokens:", response.usage.completion_tokens) 

print("Total tokens:", response.usage.total_tokens)

Let's estimate the cost of that last request. As of October 22, 2026, the [GPT-5.4 nano model page](https://developers.openai.com/api/docs/models/gpt-5.4-nano) lists standard text prices of $0.20 per million input tokens and $1.25 per million output tokens.

This simple estimate treats all input as uncached and excludes taxes or special processing rates. Check the current price if you change models; don't reuse these rates for a different one.

In [ ]:
input_price_per_million = 0.20
output_price_per_million = 1.25

input_cost = (response.usage.prompt_tokens / 1_000_000) * input_price_per_million
output_cost = (response.usage.completion_tokens / 1_000_000) * output_price_per_million
total_cost = input_cost + output_cost

print(f'Estimated cost of our last request: ${total_cost:.8f}')

A short request is inexpensive, but that doesn't mean unlimited requests are free. Start with one or two examples, check the result and usage, and only then run a larger batch. Save results so you don't pay again just because you restarted the notebook.

--- 
# 4. The OpenAI Embeddings endpoint

## 4.1 Overhead

Imports for this section:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 4.2 Vector representations of text

Say that we're trying to build a search engine and we want to return the document most relevant to the user's search query.

In [ ]:
query = "deep learning models"

documents = [
    "deep cleaning services",
    "chicago deep dish pizza",
    "artificial neural networks",
]

Let's start by turning our text into numeric data using `CountVectorizer`, which we covered earlier in `09.3 text-analytics.ipynb`:

In [ ]:
# Keep documents unchanged, so rerunning this cell doesn't keep adding the query
texts = documents + [query]

vectorizer = CountVectorizer(binary=True)
matrix = vectorizer.fit_transform(texts)

vector_df = pd.DataFrame(matrix.toarray(), index=texts,
                         columns=vectorizer.get_feature_names_out())
vector_df

In the representation above, we have a **vector** for each mini document.

- Each value in the vector corresponds to whether a particular word appears (1) or not (0), because we used `binary=True`.
- You can think of the values in the vector as coordinates in a high-dimensional space.

Now, let's take it a step further and compute the similarities between each document and the user's query. To do this, we'll use the **cosine similarity function** from scikit-learn.

- Cosine similarity is a metric that lies on the interval [-1, 1] (or [0, 1], if the elements of the vector are non-negative). 
- The geometrical interpretation of cosine similarity is that it measures the cosine of the angle between two vectors.
- The higher the value of cosine similarity, the more similar the vectors.

In [ ]:
# Calculate the similarities between the texts
cosine_similarities = cosine_similarity(matrix)

# Create a DataFrame with those similarities
cosine_similarity_df = pd.DataFrame(cosine_similarities, index=texts, columns=texts)

# Find the documents most similar to the query (other than the query itself)
cosine_similarity_df = cosine_similarity_df.loc[query].drop(query)

# Rank in order of similarity score, from highest to lowest
cosine_similarity_df = cosine_similarity_df.sort_values(ascending=False)

print("Query:", query)
print("----------------------------------------")
print(cosine_similarity_df)

What do we notice from inspecting these values?

- Even though "artificial neural networks" is conceptually closer to the user's query than "deep cleaning services" or "chicago deep dish pizza," it's ranked last in terms of similarity because there aren't any words in common.

This example presents one issue with word count representations, like the ones generated by CountVectorizer: they neglect the **meaning** of each document. 

As a result, it's harder for us to perform tasks that rely on meaning, like clustering documents related to similar topics (e.g., in order to build a recommender system).

## 4.3 OpenAI's embeddings

An embedding model turns text into a vector that can capture more about meaning than shared words alone. It isn't perfect, but let's see how the ranking changes.

We'll use `text-embedding-3-small` and send the three documents plus the query in **one request**. The query and documents need to use the same embedding model, so their vectors can be compared.

In [ ]:
def get_embeddings(model, texts):
    embedding_response = client.embeddings.create(
        model=model,
        input=texts,
        encoding_format='float',
    )
    # Each result's index tells us which input text it belongs to
    return {texts[item.index]: item.embedding for item in embedding_response.data}

Create a dictionary mapping each text to its embedding. For this tiny example, all four fit in one request. For a larger collection, use batches and save results along the way so you don't lose them or pay to recompute them.

In [ ]:
embeddings = get_embeddings(model='text-embedding-3-small', texts=texts)

By default, `text-embedding-3-small` returns a vector of 1,536 numbers per text. Let's check the lengths and look at the first five numbers. The individual numbers aren't word counts or probabilities.

[More on embedding models and dimensions](https://developers.openai.com/api/docs/guides/embeddings).

In [ ]:
for doc, embedding in embeddings.items():
  print("Document:", doc)
  print("Length of embedding vector:", len(embedding))
  print("First five elements:", embedding[:5])
  print("------------------------------")

Now, let's compute the cosine similarities between documents again, using the OpenAI embeddings.

In [ ]:
# Combine the embeddings from OpenAI into a matrix
embeddings_matrix = np.array([embeddings[text] for text in texts])

# Use the matrix to calculate the similarities between the texts
cosine_similarities = cosine_similarity(embeddings_matrix)

# Create a DataFrame with those similarities
cosine_similarity_df = pd.DataFrame(cosine_similarities, index=texts, columns=texts)

# Find the documents most similar to the query (other than the query itself)
cosine_similarity_df = cosine_similarity_df.loc[query].drop(query)

# Rank in order of similarity score, from highest to lowest
cosine_similarity_df = cosine_similarity_df.sort_values(ascending=False)

print("Query:", query)
print("----------------------------------------")
print(cosine_similarity_df)

Does "artificial neural networks" now rank above the other documents? Compare the **ranking**, not whether every similarity number went up. These are two different representations; cosine similarity is not a probability that a document is relevant.

---
---

## EXERCISE # 2

Let's compare a word-based classifier with an LLM. To make the comparison useful, they need to predict the **same 100 held-out rows**.

The setup below reads the same book reviews as `09.3`, keeps 100 rows aside (50 positive, 50 negative), and trains a TF-IDF/logistic-regression baseline on the remaining 1,900. This is a separate holdout comparison; we aren't changing the earlier cross-validation examples.

We'll keep all the original rows, including duplicate review texts. So "held out" refers to the row split, not a guarantee that the wording never appears in the training rows.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

reviews = pd.read_csv(data_folder / 'books.csv', quotechar='"', escapechar='\\')
train_rows, test_rows = train_test_split(
    reviews.index, test_size=100, random_state=42, stratify=reviews['positive']
)
train_reviews = reviews.loc[train_rows]
test_reviews = reviews.loc[test_rows].copy()

# Learn the vocabulary from the training rows; keep the test rows for evaluation
comparison_vectorizer = TfidfVectorizer()
X_train = comparison_vectorizer.fit_transform(train_reviews['review_text'])
X_test = comparison_vectorizer.transform(test_reviews['review_text'])

comparison_model = LogisticRegression(max_iter=100000)
comparison_model.fit(X_train, train_reviews['positive'])
test_reviews['baseline_prediction'] = comparison_model.predict(X_test)
baseline_accuracy = accuracy_score(test_reviews['positive'], test_reviews['baseline_prediction'])
print(f'Baseline accuracy on these 100 rows: {baseline_accuracy:.3f}')

test_reviews[['review_text', 'positive', 'baseline_prediction']].head()

Now your turn:

1. Try one or two reviews with `model_name` first. Send only the review text, not the correct `positive` label or the baseline's prediction.
2. Classify the reviews in `test_reviews` as positive or negative. Keep the row ids and save your predictions, so you don't need to pay for the same requests again.
3. Check that every answer is a valid label, then map `positive` to 1 and `negative` to 0. Don't silently drop failed requests: you need a prediction for each of the same 100 rows before comparing accuracy.
4. Compare the LLM's accuracy with `baseline_accuracy`. Which reviews do they disagree on? Does the extra cost seem worth it?

Use the bounded request settings above. If a request fails, inspect the error before retrying; don't wrap paid requests in an unlimited retry loop. Don't tune your prompt on these test labels and then treat the resulting accuracy as a fresh test.

---
---